In [ ]:
import calendar
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

ASSET_TABLE = "ehm_fleetstore_prd1eun82719_internal.assetmanagement.aircraftengine"
MASTER_TABLE = "ehm_fleetstore_prd1eun82719_internal.`g700-pearl700`.`emucontinuousmaster-aircraftengine-cda`"
SCAN_TABLE = "ehm_fleetstore_prd1eun82719_internal.`g700-pearl700`.`emucontinuousscan-cda`"
ENGINE_TYPE_CODE = "Pearl 700"
CHANNELS = ("AC", "BC")
PARAMETER_PREFIX = "Parameter:EVHMU_EEC_"

# Preserve valid widget values across reruns so changing one input does not reset the others.
def _current_widget_value(name):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return None


def _replace_dropdown(name, choices, default, label):
    string_choices = [str(choice) for choice in choices]
    if not string_choices:
        raise ValueError(f"Widget {name} requires at least one choice.")
    current = _current_widget_value(name)
    selected = current if current in string_choices else str(default)
    try:
        dbutils.widgets.remove(name)
    except Exception:
        pass
    dbutils.widgets.dropdown(name, selected, string_choices, label)
    return dbutils.widgets.get(name)


def _replace_text(name, default, label):
    current = _current_widget_value(name)
    value = current if current not in (None, "") else str(default)
    try:
        dbutils.widgets.remove(name)
    except Exception:
        pass
    dbutils.widgets.text(name, value, label)
    return dbutils.widgets.get(name)


# Normalize the asset identifiers to strings because Databricks widget values are always strings.
asset_df = (
    spark.table(ASSET_TABLE)
    .where(F.col("EngineTypeCode") == ENGINE_TYPE_CODE)
    .select(
        F.col("LatestAircraftLatestIdentifier").cast("string").alias("AircraftIdentifier"),
        F.col("LatestAircraftId").alias("AircraftId"),
        F.col("EngineSerialNumber").cast("string").alias("EngineSerialNumber"),
        F.col("EngineId").alias("EngineId"),
        F.col("LatestOperatorName").alias("OperatorName"),
    )
    .where(F.col("AircraftIdentifier").isNotNull())
    .dropDuplicates()
)

aircraft_options = [
    row.AircraftIdentifier
    for row in asset_df.select("AircraftIdentifier").distinct().orderBy("AircraftIdentifier").collect()
]
if not aircraft_options:
    raise RuntimeError(f"No aircraft were found for engine type {ENGINE_TYPE_CODE}.")

selected_acid = _replace_dropdown(
    "AircraftID",
    aircraft_options,
    aircraft_options[0],
    "Aircraft identifier",
)

# Require one current aircraft asset ID instead of silently taking an arbitrary collected row.
selected_asset_df = asset_df.where(F.col("AircraftIdentifier") == selected_acid)
internal_aircraft_ids = [
    row.AircraftId
    for row in selected_asset_df.select("AircraftId").where(F.col("AircraftId").isNotNull()).distinct().collect()
]
if len(internal_aircraft_ids) != 1:
    raise ValueError(
        f"Aircraft {selected_acid} maps to {len(internal_aircraft_ids)} current internal aircraft IDs."
    )
internal_acid = internal_aircraft_ids[0]

# Build a validated ESN-to-engine-ID map for the selected aircraft only.
engine_id_by_esn = {}
for row in (
    selected_asset_df.select("EngineSerialNumber", "EngineId")
    .where(F.col("EngineSerialNumber").isNotNull())
    .distinct()
    .collect()
):
    esn = str(row.EngineSerialNumber)
    if esn in engine_id_by_esn and engine_id_by_esn[esn] != row.EngineId:
        raise ValueError(f"Engine serial number {esn} maps to more than one engine ID.")
    engine_id_by_esn[esn] = row.EngineId

engine_options = sorted(engine_id_by_esn)
if not engine_options:
    raise RuntimeError(f"No engines were found for aircraft {selected_acid}.")

selected_esn = _replace_dropdown(
    "EngineSerialNumber",
    engine_options,
    engine_options[0],
    "Engine serial number",
)
engineid = engine_id_by_esn[selected_esn]

flight_date = _replace_text("Flight_Date", "2026-01-01", "Load flights from")
years_text = _replace_text("num_years", "2", "Years to load")

# Validate the date window and make the years control effective.
try:
    window_start_date = datetime.strptime(flight_date, "%Y-%m-%d").date()
except ValueError as exc:
    raise ValueError("Flight_Date must use YYYY-MM-DD format.") from exc

try:
    years_load = int(years_text)
except ValueError as exc:
    raise ValueError("Years to load must be a whole number.") from exc
if years_load < 1 or years_load > 20:
    raise ValueError("Years to load must be between 1 and 20.")

target_year = window_start_date.year + years_load
target_day = min(window_start_date.day, calendar.monthrange(target_year, window_start_date.month)[1])
window_end_exclusive = window_start_date.replace(year=target_year, day=target_day)
window_end_date = window_end_exclusive - timedelta(days=1)
start_calendar_id = int(window_start_date.strftime("%Y%m%d"))
end_calendar_id = int(window_end_date.strftime("%Y%m%d"))

# Filter the master data to the selected engine before resolving a flight window.
cda_master_dt = (
    spark.table(MASTER_TABLE)
    .where(
        (F.col("AircraftIdentifier").cast("string") == selected_acid)
        & (F.col("EngineSerialNumber").cast("string") == selected_esn)
        & F.col("CalendarId").between(start_calendar_id, end_calendar_id)
    )
)

flight_rows = (
    cda_master_dt.select("StartDatetime", "EndDatetime")
    .where(F.col("StartDatetime").isNotNull() & F.col("EndDatetime").isNotNull())
    .groupBy("StartDatetime")
    .agg(F.max("EndDatetime").alias("EndDatetime"))
    .where(F.col("EndDatetime") >= F.col("StartDatetime"))
    .orderBy(F.col("StartDatetime").desc())
    .collect()
)

flight_window_by_label = {
    row.StartDatetime.isoformat(sep=" "): (row.StartDatetime, row.EndDatetime)
    for row in flight_rows
}
no_flight_option = "No matching flights"
flight_options = list(flight_window_by_label) or [no_flight_option]
selected_flight_label = _replace_dropdown(
    "FlightStartDateTime",
    flight_options,
    flight_options[0],
    "Flight start time",
)
flight_available = selected_flight_label in flight_window_by_label

print(
    f"Aircraft: {selected_acid} | Internal aircraft ID: {internal_acid} | "
    f"ESN: {selected_esn} | Engine ID: {engineid}"
)
print(f"Flight search window: {window_start_date} through {window_end_date}")
print(f"Matching flights: {len(flight_rows)}")

In [ ]:
# Build every year-month partition touched by the flight so month-boundary flights are complete.
def _month_partitions_between(start_datetime, end_datetime):
    current = date(start_datetime.year, start_datetime.month, 1)
    final = date(end_datetime.year, end_datetime.month, 1)
    partitions = []
    while current <= final:
        partitions.append((current.year, current.month))
        if current.month == 12:
            current = date(current.year + 1, 1, 1)
        else:
            current = date(current.year, current.month + 1, 1)
    return partitions


scan_table = spark.table(SCAN_TABLE)
cda_cols = scan_table.columns

if flight_available:
    selected_start_dt, selected_end_dt = flight_window_by_label[selected_flight_label]
    start_dt = selected_start_dt
    end_dt = selected_end_dt
    flight_duration = end_dt - start_dt

    partition_filter = F.lit(False)
    for partition_year, partition_month in _month_partitions_between(start_dt, end_dt):
        partition_filter = partition_filter | (
            (F.col("year") == partition_year) & (F.col("Month") == partition_month)
        )

    # Restrict samples by aircraft, engine, physical timestamp, and the necessary partitions.
    cda = (
        scan_table.where(
            (F.col("AircraftId") == internal_acid)
            & (F.col("AssetIdentifier").cast("string") == selected_esn)
            & partition_filter
            & F.col("Timestamp").between(start_dt, end_dt)
        )
        .orderBy(F.col("Timestamp"))
    )
    print(f"Selected flight: {start_dt} through {end_dt} | Duration: {flight_duration}")
else:
    # Keep the downstream schema intact when the date window contains no matching flight.
    selected_start_dt = None
    selected_end_dt = None
    start_dt = None
    end_dt = None
    flight_duration = None
    cda = scan_table.limit(0)
    print("No flight is available for the current aircraft, engine, and date window.")

In [ ]:
# Define channel-neutral suffixes once, then generate matching AC and BC columns programmatically.
THRUST_REVERSER_SUFFIXES = [
    "IAcThrustReverser_tcmIntlV_data",
    "IAcThrustReverser_trLeftDoorSwLocked_LOWER__data",
    "IAcThrustReverser_trLeftDoorSwLocked_UPPER__data",
    "IAcThrustReverser_trLwrDoorLeftSwLocked_data",
    "IAcThrustReverser_trLwrDoorRightSwLocked_data",
    "IAcThrustReverser_trLwrDoorRightSwLocked_flt",
    "IAcThrustReverser_trRightDoorSwLocked_LOWER__data",
    "IAcThrustReverser_trRightDoorSwLocked_UPPER__data",
    "IAcThrustReverser_trUprDoorLeftSwLocked_data",
    "IAcThrustReverser_trUprDoorLeftSwLocked_flt",
    "IAcThrustReverser_trUprDoorRightSwLocked_data",
    "IAcThrustReverser_trUprDoorRightSwLocked_flt",
    "IOSThrustReverser_mainTestEnabled_data",
    "IOSThrustReverser_trDoorPosVa_data",
    "IOSThrustReverser_trDoorPosVb_data",
    "IOSThrustReverser_trDoorPosVex_data",
    "IOSThrustReverser_trDoorPosVex_extflt",
    "IOSThrustReverser_trDoorPosVex_intflt",
    "IOSThrustReverser_trLoSwRaw_data",
    "IOSThrustReverser_trUpSwRaw_data",
    "IOtherProcThrustReverser_trLoLVTSelected_data",
    "IOtherProcThrustReverser_trUpLVTSelected_data",
    "IThrustReverser_loOwnTRLVTRaw_data",
    "IThrustReverser_loOwnTRLVTRngFlt_data",
    "IThrustReverser_trAnomaly_data",
    "IThrustReverser_trArmed_data",
    "IThrustReverser_trDeploySelected_data",
    "IThrustReverser_trDoorDeplStatus_data",
    "IThrustReverser_trDoorPosLost_data",
    "IThrustReverser_trDoorStowStatus_data",
    "IThrustReverser_trInTransit_data",
    "IThrustReverser_trInadvDeploy_data",
    "IThrustReverser_trIsDeployed_data",
    "IThrustReverser_trIsUnlocked_data",
    "IThrustReverser_trJam_data",
    "IThrustReverser_trLVTPosVCtrl_data",
    "IThrustReverser_trLegalDeployCmd_data",
    "IThrustReverser_trLessDeplPos_data",
    "IThrustReverser_trLoLVTSelected_data",
    "IThrustReverser_trLoLVTSelected_flt",
    "IThrustReverser_trMTESTimeExpired_data",
    "IThrustReverser_trMTESV_data",
    "IThrustReverser_trMoreDeplPos_data",
    "IThrustReverser_trUnavailable_data",
    "IThrustReverser_trUpLVTSelected_data",
    "IThrustReverser_trUpLVTSelected_flt",
    "IThrustReverser_trVPrSwVFlt_data",
    "IThrustReverser_trVPrSwV_data",
    "IThrustReverser_upOwnTRLVTRaw_data",
    "IThrustReverser_upOwnTRLVTRngFlt_data",
]

THRUST_REVERSER_COLUMNS = [
    f"{PARAMETER_PREFIX}{channel}_{suffix}"
    for channel in CHANNELS
    for suffix in THRUST_REVERSER_SUFFIXES
]
CONTEXT_COLUMNS = [
    f"{PARAMETER_PREFIX}AC_IHPShaft_nhV_data",
    f"{PARAMETER_PREFIX}BC_IHPShaft_nhV_data",
    f"{PARAMETER_PREFIX}AC_IAircraftState_altitudeC_data",
    f"{PARAMETER_PREFIX}BC_IAircraftState_altitudeC_data",
]
IDENTIFIER_COLUMNS = ["Timestamp", "StartDatetime", "ParentAssetIdentifier", "AssetIdentifier"]

# Treat identifiers as mandatory while allowing optional signals to be absent from a schema revision.
scan_schema = set(cda_cols)
missing_identifiers = [column for column in IDENTIFIER_COLUMNS if column not in scan_schema]
if missing_identifiers:
    raise ValueError(f"The scan table is missing required columns: {missing_identifiers}")

requested_parameter_columns = THRUST_REVERSER_COLUMNS + CONTEXT_COLUMNS
unavailable_parameter_columns = [
    column for column in requested_parameter_columns if column not in scan_schema
]
PARAMETER_COLUMNS = [column for column in requested_parameter_columns if column in scan_schema]
selected_cols = IDENTIFIER_COLUMNS + PARAMETER_COLUMNS

if unavailable_parameter_columns:
    print(f"Unavailable optional parameters: {len(unavailable_parameter_columns)}")

# Materialize only the chosen engine and columns after Spark has applied every flight filter.
select_para_df = cda.select(*selected_cols).orderBy(F.col("Timestamp"))
_data_df = select_para_df.toPandas()

if not _data_df.empty:
    _data_df["Timestamp"] = pd.to_datetime(_data_df["Timestamp"], errors="coerce")
    _data_df["StartDatetime"] = pd.to_datetime(_data_df["StartDatetime"], errors="coerce")
    _data_df = _data_df.sort_values("Timestamp", kind="stable").reset_index(drop=True)

print(f"Loaded samples: {len(_data_df):,}")
print(f"Loaded parameter columns: {len(PARAMETER_COLUMNS)}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Keep short names in the UI while retaining an unambiguous mapping to the Spark columns.
def _short_parameter_name(column):
    return column[len(PARAMETER_PREFIX):] if column.startswith(PARAMETER_PREFIX) else column


combined_channel_columns = [_short_parameter_name(column) for column in PARAMETER_COLUMNS]
FEATURE_COLUMNS = [
    _short_parameter_name(column)
    for column in CONTEXT_COLUMNS
    if column in PARAMETER_COLUMNS
]

input_search = widgets.Combobox(
    placeholder="Search and add a parameter",
    options=[option for option in combined_channel_columns if option not in FEATURE_COLUMNS],
    description="Add input:",
    ensure_option=True,
    layout=widgets.Layout(width="650px"),
)
input_display = widgets.SelectMultiple(
    options=tuple(FEATURE_COLUMNS),
    description="Plot inputs:",
    layout=widgets.Layout(width="650px", height="280px"),
)
remove_input_btn = widgets.Button(
    description="Remove highlighted",
    button_style="danger",
    icon="trash",
)
apply_input_btn = widgets.Button(
    description="Apply inputs",
    button_style="info",
)
selected_output = widgets.Output()

# Mutate this list in place so plotting callbacks always see the latest applied inputs.
selected_parameters = list(FEATURE_COLUMNS)


def _refresh_search_options():
    active = set(input_display.options)
    input_search.options = [
        option for option in combined_channel_columns if option not in active
    ]


def _sync_selected_parameters():
    selected_parameters[:] = list(input_display.options)


def add_to_input(change):
    value = change["new"]
    if value and value in combined_channel_columns and value not in input_display.options:
        input_display.options = tuple(input_display.options) + (value,)
        _sync_selected_parameters()
        input_search.value = ""
        _refresh_search_options()


def remove_input(_):
    highlighted = set(input_display.value)
    input_display.options = tuple(
        option for option in input_display.options if option not in highlighted
    )
    _sync_selected_parameters()
    _refresh_search_options()


def apply_inputs(_):
    _sync_selected_parameters()
    with selected_output:
        selected_output.clear_output()
        print(f"Applied {len(selected_parameters)} plot inputs.")


input_search.observe(add_to_input, names="value")
remove_input_btn.on_click(remove_input)
apply_input_btn.on_click(apply_inputs)

input_column = widgets.VBox(
    [input_search, input_display, remove_input_btn, apply_input_btn, selected_output]
)
display(input_column)

In [ ]:
from collections import OrderedDict

import ipywidgets as widgets
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from ipywidgets import SelectionRangeSlider

# Reapply the ESN filter defensively in case _data_df was supplied by an earlier notebook run.
data_df = _data_df.copy()
if "AssetIdentifier" in data_df.columns:
    data_df = data_df.loc[
        data_df["AssetIdentifier"].astype("string") == str(selected_esn)
    ].copy()
if "Timestamp" in data_df.columns:
    data_df["Timestamp"] = pd.to_datetime(data_df["Timestamp"], errors="coerce")
    data_df = data_df.dropna(subset=["Timestamp"]).sort_values("Timestamp", kind="stable")

MAX_SLIDER_POINTS = 1000
MAX_SIGNAL_GROUPS = 16
CHANNEL_COLORS = {"AC": "#1f77b4", "BC": "#ff7f0e"}


def _full_parameter_name(short_name):
    return short_name if short_name.startswith(PARAMETER_PREFIX) else PARAMETER_PREFIX + short_name


def _channel_and_signal(short_name):
    for channel in CHANNELS:
        prefix = f"{channel}_"
        if short_name.startswith(prefix):
            return channel, short_name[len(prefix):]
    return "", short_name


def _numeric_series(series):
    non_null = series.dropna()
    if non_null.empty:
        return pd.Series(np.nan, index=series.index, dtype="float64"), False
    is_boolean = non_null.map(lambda value: isinstance(value, (bool, np.bool_))).all()
    if is_boolean:
        return series.map({True: 1.0, False: 0.0}), True
    numeric = pd.to_numeric(series, errors="coerce")
    values = numeric.dropna().to_numpy(dtype=float)
    is_discrete = bool(
        len(values)
        and len(np.unique(values)) <= 8
        and np.allclose(values, np.round(values))
    )
    return numeric, is_discrete


def _match_timestamp_timezone(value, reference):
    timestamp = pd.Timestamp(value)
    if reference.tzinfo is None and timestamp.tzinfo is not None:
        return timestamp.tz_localize(None)
    if reference.tzinfo is not None and timestamp.tzinfo is None:
        return timestamp.tz_localize(reference.tzinfo)
    return timestamp


def _plot_parameters(start_time, end_time):
    chosen = list(selected_parameters)
    if not chosen:
        print("No plot inputs are applied. Add at least one parameter in the previous cell.")
        return

    grouped = OrderedDict()
    for short_name in chosen:
        channel, signal = _channel_and_signal(short_name)
        grouped.setdefault(signal, []).append((channel, short_name))

    if len(grouped) > MAX_SIGNAL_GROUPS:
        print(
            f"The current selection creates {len(grouped)} signal groups. "
            f"Reduce it to {MAX_SIGNAL_GROUPS} or fewer for a readable plot."
        )
        return

    visible = data_df.loc[
        data_df["Timestamp"].between(start_time, end_time, inclusive="both")
    ].copy()
    if visible.empty:
        print("No samples fall inside the requested time range.")
        return

    figure_height = max(4.5, 3.0 * len(grouped))
    figure, axes = plt.subplots(
        len(grouped),
        1,
        figsize=(22, figure_height),
        sharex=True,
        squeeze=False,
    )
    skipped = []

    # Put AC and BC versions of the same signal on one axis for direct channel comparison.
    for axis, (signal, members) in zip(axes[:, 0], grouped.items()):
        plotted = False
        for channel, short_name in members:
            full_name = _full_parameter_name(short_name)
            if full_name not in visible.columns:
                skipped.append(short_name)
                continue

            values, is_discrete = _numeric_series(visible[full_name])
            valid = values.notna()
            if not valid.any():
                skipped.append(short_name)
                continue

            axis.plot(
                visible.loc[valid, "Timestamp"],
                values.loc[valid],
                color=CHANNEL_COLORS.get(channel, "#333333"),
                linewidth=1.2,
                drawstyle="steps-post" if is_discrete else "default",
                label=channel or short_name,
            )
            plotted = True

        axis.set_ylabel(signal, fontsize=9)
        axis.grid(True, alpha=0.25)
        if plotted:
            axis.legend(loc="upper right", ncol=max(1, len(members)))
        else:
            axis.text(0.5, 0.5, "No numeric samples", ha="center", va="center", transform=axis.transAxes)

    axes[-1, 0].set_xlabel("Timestamp")
    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    figure.suptitle(
        f"Aircraft {selected_acid} | Engine {selected_esn} | "
        f"{start_time} through {end_time}",
        fontsize=14,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.98))
    plt.show()
    plt.close(figure)

    if skipped:
        print("Skipped parameters without numeric samples: " + ", ".join(sorted(set(skipped))))


if data_df.empty:
    print("No samples are available for the selected flight and engine.")
else:
    all_timestamps = [pd.Timestamp(value) for value in sorted(data_df["Timestamp"].unique())]
    if len(all_timestamps) > MAX_SLIDER_POINTS:
        indices = np.linspace(
            0,
            len(all_timestamps) - 1,
            num=MAX_SLIDER_POINTS,
            dtype=int,
        )
        slider_timestamps = [all_timestamps[index] for index in np.unique(indices)]
    else:
        slider_timestamps = all_timestamps

    def _timestamp_label(timestamp):
        return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f").rstrip("0").rstrip(".")


    time_slider = SelectionRangeSlider(
        options=[(_timestamp_label(timestamp), timestamp) for timestamp in slider_timestamps],
        index=(0, len(slider_timestamps) - 1),
        description="Time range:",
        orientation="horizontal",
        layout=widgets.Layout(width="1200px"),
        continuous_update=False,
    )
    txt_start = widgets.Text(
        value=_timestamp_label(all_timestamps[0]),
        description="Start:",
        layout=widgets.Layout(width="500px"),
    )
    txt_end = widgets.Text(
        value=_timestamp_label(all_timestamps[-1]),
        description="End:",
        layout=widgets.Layout(width="500px"),
    )
    plot_btn = widgets.Button(
        description="Plot",
        button_style="success",
        icon="line-chart",
        layout=widgets.Layout(width="150px"),
    )
    plot_output = widgets.Output()

    def _render_range(start_time, end_time):
        with plot_output:
            clear_output(wait=True)
            if start_time > end_time:
                start_time, end_time = end_time, start_time
            _plot_parameters(start_time, end_time)


    def _on_plot_click(_):
        reference = all_timestamps[0]
        try:
            start_time = _match_timestamp_timezone(txt_start.value, reference)
            end_time = _match_timestamp_timezone(txt_end.value, reference)
        except Exception:
            with plot_output:
                clear_output(wait=True)
                print("Start and end must be valid date-time values.")
            return
        _render_range(start_time, end_time)


    def _on_slider_change(change):
        start_time, end_time = map(pd.Timestamp, change["new"])
        txt_start.value = _timestamp_label(start_time)
        txt_end.value = _timestamp_label(end_time)
        _render_range(start_time, end_time)


    plot_btn.on_click(_on_plot_click)
    time_slider.observe(_on_slider_change, names="value")

    controls = widgets.VBox([txt_start, txt_end, time_slider, plot_btn])
    display(widgets.VBox([controls, plot_output]))
    _render_range(all_timestamps[0], all_timestamps[-1])

In [ ]:
# Verify that every requested thrust-reverser suffix has matching AC and BC schema coverage.
schema_columns = set(cda_cols)
selected_parameter_set = set(selected_parameters)
channel_pair_records = []

for suffix in THRUST_REVERSER_SUFFIXES:
    ac_short = f"AC_{suffix}"
    bc_short = f"BC_{suffix}"
    ac_full = PARAMETER_PREFIX + ac_short
    bc_full = PARAMETER_PREFIX + bc_short
    channel_pair_records.append(
        {
            "Signal": suffix,
            "AC available": ac_full in schema_columns,
            "BC available": bc_full in schema_columns,
            "AC selected": ac_short in selected_parameter_set,
            "BC selected": bc_short in selected_parameter_set,
        }
    )

channel_pair_df = pd.DataFrame(channel_pair_records)
unpaired_schema_rows = channel_pair_df.loc[
    channel_pair_df["AC available"] != channel_pair_df["BC available"]
]

print(f"Thrust-reverser signal pairs: {len(channel_pair_df)}")
print(f"Schema pairs with unequal AC/BC availability: {len(unpaired_schema_rows)}")
display(channel_pair_df)

In [ ]:
# Keep the detailed flight inventory read-only and tied to the active widget selections.
diagnostic_master_columns = [
    "AircraftIdentifier",
    "EngineSerialNumber",
    "AircraftId",
    "EngineId",
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "EnginePosition",
    "OperatorId",
    "OperatorCode",
    "LastGeneratedDatetime",
    "FirstGeneratedDatetime",
    "Changed",
    "Created",
    "Migrated",
    "CalendarId",
]

diagnostic_master_with_duration = cda_master_dt.withColumn(
    "Duration",
    F.col("EndDatetime") - F.col("StartDatetime"),
)
diagnostic_available_columns = [
    column
    for column in diagnostic_master_columns
    if column in diagnostic_master_with_duration.columns
]
diagnostic_master_df = (
    diagnostic_master_with_duration.select(*diagnostic_available_columns)
    .orderBy(F.col("StartDatetime").desc(), F.col("EndDatetime").desc())
)

print(
    f"Flight inventory for aircraft {selected_acid}, engine {selected_esn}, "
    f"from {window_start_date} through {window_end_date}"
)
display(diagnostic_master_df)

In [ ]:
# Show the exact master record used for the active flight without positional row indexing.
if flight_available:
    diagnostic_selected_master_df = (
        diagnostic_master_df.where(F.col("StartDatetime") == selected_start_dt)
        .orderBy(F.col("EndDatetime").desc())
    )
    print(f"Selected master records for flight starting at {selected_start_dt}")
else:
    diagnostic_selected_master_df = diagnostic_master_df.limit(0)
    print("No selected flight master record is available.")

display(diagnostic_selected_master_df)

In [ ]:
# Quantify AC/BC population and exact disagreement without assuming an engineering tolerance.
channel_coverage_records = []
for suffix in THRUST_REVERSER_SUFFIXES:
    ac_column = f"{PARAMETER_PREFIX}AC_{suffix}"
    bc_column = f"{PARAMETER_PREFIX}BC_{suffix}"
    if ac_column not in _data_df.columns or bc_column not in _data_df.columns:
        continue

    ac_values = _data_df[ac_column]
    bc_values = _data_df[bc_column]
    both_present = ac_values.notna() & bc_values.notna()
    exact_disagreement = (
        ac_values.loc[both_present].ne(bc_values.loc[both_present]).fillna(False)
    )
    channel_coverage_records.append(
        {
            "Signal": suffix,
            "AC non-null": int(ac_values.notna().sum()),
            "BC non-null": int(bc_values.notna().sum()),
            "Both non-null": int(both_present.sum()),
            "Exact disagreements": int(exact_disagreement.sum()),
        }
    )

channel_coverage_df = pd.DataFrame(channel_coverage_records)
if channel_coverage_df.empty:
    print("No paired AC/BC samples are available for coverage analysis.")
else:
    print(f"Channel coverage calculated from {len(_data_df):,} flight samples.")
display(channel_coverage_df)

In [ ]:
# Preview the active flight deterministically without overwriting notebook-wide identifiers.
preview_identifier_columns = [
    column
    for column in IDENTIFIER_COLUMNS
    if column in select_para_df.columns
]
preview_parameter_columns = [
    _full_parameter_name(short_name)
    for short_name in selected_parameters
    if _full_parameter_name(short_name) in select_para_df.columns
]
preview_columns = preview_identifier_columns + [
    column
    for column in preview_parameter_columns
    if column not in preview_identifier_columns
]

diagnostic_preview_df = (
    select_para_df.select(*preview_columns)
    .orderBy(F.col("Timestamp"))
    .limit(50)
)
print(
    f"First 50 ordered samples for aircraft {selected_acid}, engine {selected_esn}, "
    f"flight start {selected_start_dt}"
)
display(diagnostic_preview_df)